6_Credit_Approval

¿Quién lo creó y cómo se obtuvieron los datos?
El dataset fue enviado por J.R. Quinlan de la Universidad de Sydney al UCI Machine Learning Repository en 1987, siendo referenciado en su trabajo "Simplifying decision trees", publicado en International Journal of Man-Machine Studies en diciembre de 1987. Los datos provienen de solicitudes de tarjetas de crédito de una institución financiera real, cuya identidad permanece anónima por razones de confidencialidad. La fuente real de los datos nunca ha sido revelada públicamente.

¿De qué trata?
Este archivo contiene solicitudes de tarjetas de crédito. Todos los nombres de atributos y valores han sido cambiados a símbolos sin significado para proteger la confidencialidad de los datos. Este dataset es interesante porque tiene una buena mezcla de atributos: continuos, nominales con pocos valores, y nominales con más valores. También tiene algunos valores faltantes. 

¿Qué contiene?
Contiene 690 instancias y 15 atributos más el atributo de clase. Las variables (anonimizadas) van de A1 a A15: A1 es binaria (b, a), A2 es continua, A3 continua, A4 categórica con 4 niveles, A5 categórica con 3 niveles, A6 categórica con 14 niveles, A7 categórica con 9 niveles, A8 continua, A9 binaria (t/f), A10 binaria (t/f), A11 continua, A12 binaria (t/f), A13 categórica con 3 niveles, A14 continua y A15 continua. La variable de clase A16 es (+) para aprobado o (−) para rechazado. Aunque los nombres están anonimizados, estudios del dataset han sugerido que corresponden a variables como edad, historial crediticio, ingresos, empleo y estado civil.

Objetivo del modelo
Clasificación binaria: predecir si una solicitud de tarjeta de crédito será aprobada (+) o rechazada (−). Requiere manejar valores faltantes, codificar variables categóricas y trabajar con una mezcla de tipos de datos. Es ideal para practicar pipelines completos de preprocesamiento y algoritmos como Logistic Regression, Decision Trees y k-NN. Las clases están relativamente balanceadas (~55% aprobadas, ~45% rechazadas).

In [1]:
# ============================================================
# LIBRERÍAS GENERALES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [2]:
# ── PASO 1: CARGA ───────────────────────────────────────────
nombres_cr = ['A1','A2','A3','A4','A5','A6','A7','A8',
              'A9','A10','A11','A12','A13','A14','A15','A16']

df_credit = pd.read_csv('Datasets/6_Credit_Approval/crx.data',
                         names=nombres_cr,
                         na_values='?')   # '?' pasa a ser NaN

print('Shape:', df_credit.shape)
print('\nNulos:')
print(df_credit.isnull().sum())
print('\nTarget A16:')
print(df_credit['A16'].value_counts())

Shape: (690, 16)

Nulos:
A1     12
A2     12
A3      0
A4      6
A5      6
A6      9
A7      9
A8      0
A9      0
A10     0
A11     0
A12     0
A13     0
A14    13
A15     0
A16     0
dtype: int64

Target A16:
A16
-    383
+    307
Name: count, dtype: int64


In [3]:
# ── PASO 2: LIMPIEZA ────────────────────────────────────────
# Columnas de texto: rellenar nulos con moda
for col in df_credit.select_dtypes(include='object').columns:
    df_credit[col] = df_credit[col].fillna(df_credit[col].mode()[0])

# Columnas numéricas: rellenar nulos con mediana
for col in df_credit.select_dtypes(include='number').columns:
    df_credit[col] = df_credit[col].fillna(df_credit[col].median())

print('Nulos restantes:', df_credit.isnull().sum().sum())

Nulos restantes: 0


In [4]:
# ── PASO 3: CONVERTIR TEXTO A NÚMEROS ───────────────────────
# Target: + = 1, - = 0
y_credit = (df_credit['A16'] == '+').values.astype(float)

# Variables categóricas → códigos numéricos
for col in ['A1','A4','A5','A6','A7','A9','A10','A12','A13']:
    df_credit[col] = pd.Categorical(df_credit[col]).codes

cols_features_cr = ['A1','A2','A3','A4','A5','A6','A7',
                    'A8','A9','A10','A11','A12','A13','A14','A15']

X_raw_cr = df_credit[cols_features_cr].values.astype(float)
m_cr = y_credit.size

print('X shape:', X_raw_cr.shape)
print('Aprobado (+):', int(y_credit.sum()), '| Rechazado (-):', int((y_credit==0).sum()))

X shape: (690, 15)
Aprobado (+): 307 | Rechazado (-): 383


In [5]:
# ============================================================
# FUNCIÓN DE BALANCEO — oversampling con numpy
# ============================================================
def balancear(X, y):
    """
    Balancea un dataset desbalanceado usando OVERSAMPLING.
    
    ¿Qué hace?
    - Identifica cuántos ejemplos tiene cada clase
    - La clase con MÁS ejemplos queda igual
    - Las clases con MENOS ejemplos se repiten (con reemplazo)
      hasta tener la misma cantidad que la clase mayoritaria
    - Al final todas las clases tienen el mismo número de filas
    
    ¿Por qué oversampling y no undersampling?
    - Undersampling borra filas → perdemos información
    - Oversampling agrega filas → mantenemos toda la información original
    """
    clases = np.unique(y)
    n_max  = max(np.sum(y == c) for c in clases)   # tamaño de la clase más grande
    
    X_bal_list = []
    y_bal_list = []
    
    for c in clases:
        idx    = np.where(y == c)[0]               # índices de esta clase
        n_c    = len(idx)                           # cuántos ejemplos tiene
        
        if n_c < n_max:
            # repetir filas hasta alcanzar n_max
            extra  = n_max - n_c
            idx_extra = np.random.choice(idx, size=extra, replace=True)
            idx_final = np.concatenate([idx, idx_extra])
        else:
            idx_final = idx
        
        X_bal_list.append(X[idx_final])
        y_bal_list.append(y[idx_final])
    
    X_bal = np.concatenate(X_bal_list, axis=0)
    y_bal = np.concatenate(y_bal_list, axis=0)
    
    # Mezclar aleatoriamente para no dejar todas las clases juntas
    perm  = np.random.permutation(len(y_bal))
    return X_bal[perm], y_bal[perm]

def mostrar_balance(y, nombre, antes_despues='ANTES'):
    """Imprime cuántos ejemplos tiene cada clase."""
    clases, cuentas = np.unique(y, return_counts=True)
    print(f'  Balance {antes_despues} — {nombre}:')
    for c, n in zip(clases, cuentas):
        print(f'    Clase {int(c)}: {n} ({n/len(y)*100:.1f}%)')

np.random.seed(42)   # para reproducibilidad
print('Funciones de balanceo definidas')

Funciones de balanceo definidas


In [6]:
mostrar_balance(y_credit, 'Credit Approval', 'ANTES')

  Balance ANTES — Credit Approval:
    Clase 0: 383 (55.5%)
    Clase 1: 307 (44.5%)


In [7]:
def featureNormalize(X):
    """
    Normaliza las features de X.
    Para cada columna: resta la media y divide por la desviación estándar.
    
    Retorna:
      X_norm : X normalizado (mismo tamaño que X)
      mu     : media de cada columna (se guarda para normalizar datos nuevos)
      sigma  : desviación estándar de cada columna
    """
    X_norm = X.copy()
    mu     = np.mean(X, axis=0)   # media de cada columna
    sigma  = np.std(X, axis=0)    # desviación estándar de cada columna
    X_norm = (X - mu) / sigma     # estandarización Z-score
    return X_norm, mu, sigma

In [8]:
X_norm_cr, mu_cr, sigma_cr = featureNormalize(X_raw_cr)
X_bal_cr, y_bal_cr = balancear(X_norm_cr, y_credit)
mostrar_balance(y_bal_cr, 'Credit Approval', 'DESPUÉS')

X_credit = np.concatenate([np.ones((len(y_bal_cr), 1)), X_bal_cr], axis=1)
y_credit  = y_bal_cr

print(f'\nX_credit: {X_credit.shape} | y_credit: {y_credit.shape}')

  Balance DESPUÉS — Credit Approval:
    Clase 0: 383 (50.0%)
    Clase 1: 383 (50.0%)

X_credit: (766, 16) | y_credit: (766,)
